# University Regulation RAG Assistant
## Google Colab Generative AI Project

**Selected topic:** University Regulation RAG Assistant

Pipeline: PDF upload → text extraction → chunking → embeddings → FAISS semantic search → similarity threshold → grounded generation → document/page citation.

The system answers only from the uploaded university documents and reports when supporting information is not found.


In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate gradio

In [ ]:
import os, re
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from google.colab import files


## 1. Upload multiple PDF documents

Upload university regulations, examination rules, attendance policies, student handbooks, curriculum rules, etc.


In [ ]:
uploaded = files.upload()
pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("Uploaded PDFs:", pdf_paths)


Saving R22 Regulations for B.Tech.pdf to R22 Regulations for B.Tech.pdf
Uploaded PDFs: ['R22 Regulations for B.Tech.pdf']


## 2. Extract PDF text with source metadata

Every extracted page keeps its document name and page number so that answers can cite their source.


In [ ]:
def extract_pages(pdf_paths):
    records = []
    for pdf_path in pdf_paths:
        reader = PdfReader(pdf_path)
        for page_number, page in enumerate(reader.pages, start=1):
            text = re.sub(r"\\s+", " ", page.extract_text() or "").strip()
            if text:
                records.append({
                    "text": text,
                    "source": os.path.basename(pdf_path),
                    "page": page_number
                })
    return records

page_records = extract_pages(pdf_paths)
print(f"Extracted {len(page_records)} non-empty pages.")
display(pd.DataFrame(page_records))

Extracted 38 non-empty pages.


,text,source,page
0,Academic\nRegulationsR22\nIn Compliance with N...,R22 Regulations for B.Tech.pdf,1
1,PREFACE\n‘You are born to Blossom’ – What an i...,R22 Regulations for B.Tech.pdf,3
2,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,4
3,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,5
4,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,7
5,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,8
6,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,9
7,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,10
8,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,11
9,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech.pdf,12


## 3. Chunking

We use 500-character chunks with 100-character overlap. Overlap helps preserve information that crosses chunk boundaries.


In [ ]:
CHUNK_SIZE = 700
CHUNK_OVERLAP = 150

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    # Split by double newline to preserve natural paragraphs
    paragraphs = text.split('\n\n')

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        if len(para) <= chunk_size:
            chunks.append(para)
        else:
            # Fixed-size sliding window with overlap for long paragraphs
            start = 0
            while start < len(para):
                end = min(start + chunk_size, len(para))
                part = para[start:end].strip()
                if part:
                    chunks.append(part)
                if end >= len(para):
                    break
                start = end - overlap
    return chunks

print(f"Using improved chunking strategy: CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")

chunks = []
for rec in page_records:
    for part in chunk_text(rec["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({
            "chunk_id": len(chunks),
            "text": part,
            "source": rec["source"],
            "page": rec["page"]
        })

print("Total chunks:", len(chunks))

# Add the definition of EMBEDDING_MODEL and embedder here
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

# Re-embed and update the FAISS index with new chunks
texts = [x["text"] for x in chunks]
embeddings = embedder.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS Index updated with", index.ntotal, "new context chunks.")

Using improved chunking strategy: CHUNK_SIZE=700, CHUNK_OVERLAP=150
Total chunks: 210


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index updated with 210 new context chunks.


## 4. Embeddings + FAISS vector store

`all-MiniLM-L6-v2` converts chunks into vectors. We normalize vectors and use FAISS inner-product search, which corresponds to cosine similarity for normalized vectors.


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = [x["text"] for x in chunks]
embeddings = embedder.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding dimension:", embeddings.shape[1])
print("Vectors in FAISS:", index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embedding dimension: 384
Vectors in FAISS: 210


## 5. Semantic retrieval

`TOP_K` controls retrieval depth. `SIMILARITY_THRESHOLD` prevents weak matches from being sent to the generator.


In [ ]:
TOP_K = 5
SIMILARITY_THRESHOLD = 0.30

def retrieve(query, top_k=TOP_K, threshold=SIMILARITY_THRESHOLD):
    q = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q, top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0 and float(score) >= threshold:
            item = chunks[int(idx)].copy()
            item["score"] = float(score)
            results.append(item)
    return results

results = retrieve("What is the minimum attendance required for examinations?")
for r in results:
    print(f"[{r['score']:.3f}] {r['source']} — page {r['page']}")
    print(r["text"])
    print("-"*70)


[0.563] R22 Regulations for B.Tech.pdf — page 17
criteria
 To be declared successful in a course, a student must secure at least a grade 4.0 in a scale 
of 10 based on the total maximum marks which is inclusive of formative and summative 
assessment. The students should also get 35% from the maximum marks allotted for formative 
and summative assessments individually.
 The hierarchy of qualifying criteria is as follows:
i. attendance compliance should be 75% or within condonable range; else the candidate 
is put into ‘R’ grade.
ii. In formative assessment, a candidate should secure a minimum of 35% i.e. 21 marks 
out of 60; else the candidate is put into ‘R’ grade.
iii. In summative assessment, a candidate should secure a minimum of 35% i
----------------------------------------------------------------------
[0.539] R22 Regulations for B.Tech.pdf — page 27
ents 
of a student’s performance in a course. 
Table	9: grading information
Relative grading Range (P) Category grade (g)
≥ 9.50 Ou

## 6. Local grounded generation model

FLAN-T5 runs in Colab without requiring a paid API. The prompt instructs it to use only retrieved context.


In [ ]:
GENERATION_MODEL = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)

def custom_generator(prompt_text, max_new_tokens=220, do_sample=False):
    input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    outputs = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": generated_text}]

generator = custom_generator
print("Generator ready.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready.


## 7. Complete RAG pipeline

If no retrieved chunk crosses the threshold, the system returns an explicit "Information not found" message.


In [ ]:
def build_context(results):
    return "\n\n".join(
        f"[SOURCE {i}] Document: {r['source']} | Page: {r['page']}\n{r['text']}"
        for i, r in enumerate(results, 1)
    )

def answer_question(question):
    question = question.strip()
    if not question:
        return "Please enter a question.", []

    results = retrieve(question)
    if not results:
        return (
            "Information not found in the uploaded university documents.",
            []
        )

    context = build_context(results)
    prompt = """You are a university regulation question-answering assistant.

Answer the QUESTION using ONLY the CONTEXT.
Do not use outside knowledge.
Do not invent rules, dates, numbers, requirements, or exceptions.
If the context does not clearly contain the answer, say:
Information not found in the uploaded university documents.

QUESTION:
%s

CONTEXT:
%s

ANSWER:""" % (question, context)

    answer = generator(prompt)[0]["generated_text"].strip()

    citations = []
    seen = set()
    for r in results:
        key = (r["source"], r["page"])
        if key not in seen:
            citations.append({
                "document": r["source"],
                "page": r["page"],
                "similarity": round(r["score"], 3)
            })
            seen.add(key)
    return answer, citations


In [ ]:
question = "What is the minimum attendance required for the end-semester examination?"
answer, citations = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\nSOURCES:")
for c in citations:
    print(f"- {c['document']} — Page {c['page']} — similarity {c['similarity']}")


## 8. Gradio chatbot interface

Run this cell to demonstrate the finished project.


In [ ]:
import gradio as gr

def chat_fn(question):
    answer, citations = answer_question(question)
    if citations:
        source_text = "\n".join(
            f"• {c['document']} — Page {c['page']} — similarity {c['similarity']}"
            for c in citations
        )
    else:
        source_text = "No supporting source was found."
    return answer, source_text

demo = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(
        label="Ask about university regulations",
        placeholder="Example: What is the minimum attendance required?"
    ),
    outputs=[
        gr.Textbox(label="Grounded Answer"),
        gr.Textbox(label="Source Document / Page")
    ],
    title="University Regulation RAG Assistant",
    description="Answers are generated from the uploaded university PDFs only.",
    examples=[
        ["What is the minimum attendance required for examinations?"],
        ["What documents must students carry to the examination hall?"],
        ["What are the credits for a course?"],
        ["What is the attendance requirement?"]
    ]
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc1864c19006c90fc5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
